# 02 · MNIST model configuration

Notebook này đọc dataset `.npz`, chia train/validation, fit StandardScaler và tìm cấu hình tốt cho Logistic Regression cùng ANN tuned.

Kết quả được lưu ở `mnist_model_config.json`.

In [1]:
from pathlib import Path
import sys
import json
import warnings
import numpy as np

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / 'gk' / 'web' / 'backend').exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from sklearn.metrics import accuracy_score
from sklearn.exceptions import ConvergenceWarning
from sklearn.preprocessing import StandardScaler
from gk.web.backend.digits_models import (
    LOGISTIC_CANDIDATES, MLP_SEARCH_CONFIGS, RANDOM_SEED, SEARCH_MAX_ITER,
    build_logistic, build_mlp, choose_search_subset,
)
warnings.filterwarnings('ignore', category=ConvergenceWarning)

ARTIFACT_DIR = PROJECT_ROOT / 'gk' / 'web' / 'backend' / 'artifacts'
DATASET_DIR = ARTIFACT_DIR / 'mnist_dataset'
DATASET_PATH = DATASET_DIR / 'mnist_normalized.npz'
SPLIT_PATH = DATASET_DIR / 'split.json'
CONFIG_PATH = DATASET_DIR / 'mnist_model_config.json'
data = np.load(DATASET_PATH)
X_train_images = data['X_train_pixels'].astype(float)
y_train = data['y_train'].astype(int)
X_test_images = data['X_test_pixels'].astype(float)
y_test = data['y_test'].astype(int)
X_train_pixels = X_train_images.reshape(len(X_train_images), -1)
X_test_pixels = X_test_images.reshape(len(X_test_images), -1)
print('Loaded:', DATASET_PATH)

Loaded: /Users/nguyenkz/Documents/code/hoc/CS2207.26.1.CH.02/gk/web/backend/artifacts/mnist_dataset/mnist_normalized.npz


In [2]:
split = json.loads(SPLIT_PATH.read_text(encoding='utf-8'))
train_indices = np.asarray(split['fit_indices'], dtype=int)
validation_indices = np.asarray(split['validation_indices'], dtype=int)
scaler = StandardScaler().fit(X_train_pixels[train_indices])
X_fit = scaler.transform(X_train_pixels[train_indices])
X_validation = scaler.transform(X_train_pixels[validation_indices])
X_test = scaler.transform(X_test_pixels)
y_fit = y_train[train_indices]
y_validation = y_train[validation_indices]
print(X_fit.shape, X_validation.shape, X_test.shape)

(48000, 64) (12000, 64) (10000, 64)


## Cấu hình bốn model

Sửa cell này để thử architecture, activation, learning rate, số epoch hoặc `C`.

In [3]:
MODEL_CONFIGS = {
    'logistic': {'kind': 'linear', 'C': 1.0, 'solver': 'lbfgs', 'max_iter': 2000},
    'mlp_4_one_layer': {
        'kind': 'ann', 'hidden_layer_sizes': [4], 'activation': 'tanh',
        'learning_rate_init': 0.001, 'max_iter': 350, 'early_stopping': False,
    },
    'mlp_4_two_layers': {
        'kind': 'ann', 'hidden_layer_sizes': [4, 4], 'activation': 'tanh',
        'learning_rate_init': 0.001, 'max_iter': 350, 'early_stopping': False,
    },
    'compact_tuned': {
        'kind': 'ann', 'hidden_layer_sizes': [48], 'activation': 'relu',
        'learning_rate_init': 0.002, 'max_iter': 350, 'early_stopping': False,
    },
}
print(MODEL_CONFIGS)

{'logistic': {'kind': 'linear', 'C': 1.0, 'solver': 'lbfgs', 'max_iter': 2000}, 'mlp_4_one_layer': {'kind': 'ann', 'hidden_layer_sizes': [4], 'activation': 'tanh', 'learning_rate_init': 0.001, 'max_iter': 350, 'early_stopping': False}, 'mlp_4_two_layers': {'kind': 'ann', 'hidden_layer_sizes': [4, 4], 'activation': 'tanh', 'learning_rate_init': 0.001, 'max_iter': 350, 'early_stopping': False}, 'compact_tuned': {'kind': 'ann', 'hidden_layer_sizes': [48], 'activation': 'relu', 'learning_rate_init': 0.002, 'max_iter': 350, 'early_stopping': False}}


## Tuning 30 vòng

Candidate được train trên tối đa 24.000 mẫu train để thử nghiệm nhanh. Bước search có giới hạn iteration riêng; warning hội tụ được ẩn khỏi output và trạng thái `converged`/`iterations` được lưu trong JSON. Cấu hình cuối sẽ được train lại trên toàn bộ train ở notebook tiếp theo.

In [ ]:
X_search, y_search, search_count = choose_search_subset(X_fit, y_fit)

logistic_scores = []
for C in LOGISTIC_CANDIDATES:
    candidate = build_logistic(float(C)).fit(X_search, y_search)
    score = accuracy_score(y_validation, candidate.predict(X_validation))
    logistic_scores.append({'C': float(C), 'validation_accuracy': float(score)})
best_logistic = sorted(logistic_scores, key=lambda row: (-row['validation_accuracy'], row['C']))[0]
MODEL_CONFIGS['logistic']['C'] = best_logistic['C']

mlp_scores = []
for hidden_layers, activation, learning_rate in MLP_SEARCH_CONFIGS:
    candidate = build_mlp(hidden_layers, activation, learning_rate, max_iter=SEARCH_MAX_ITER).fit(X_search, y_search)
    score = accuracy_score(y_validation, candidate.predict(X_validation))
    parameter_count = sum((left * right) + right for left, right in zip((64, *hidden_layers), (*hidden_layers, 10)))
    mlp_scores.append({
        'architecture': list(hidden_layers), 'activation': activation,
        'learning_rate_init': learning_rate, 'parameter_count': parameter_count,
        'iterations': int(candidate.n_iter_), 'converged': int(candidate.n_iter_) < SEARCH_MAX_ITER,
        'validation_accuracy': float(score),
    })
best_mlp = sorted(mlp_scores, key=lambda row: (-row['validation_accuracy'], row['parameter_count'], len(row['architecture'])))[0]
MODEL_CONFIGS['compact_tuned'].update({
    'hidden_layer_sizes': best_mlp['architecture'],
    'activation': best_mlp['activation'],
    'learning_rate_init': best_mlp['learning_rate_init'],
})
print('Best Logistic:', best_logistic)
print('Best ANN:', best_mlp)
print('ANN rounds:', len(mlp_scores))

/Users/nguyenkz/Documents/code/hoc/CS2207.26.1.CH.02/.venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (120) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/nguyenkz/Documents/code/hoc/CS2207.26.1.CH.02/.venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (120) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/nguyenkz/Documents/code/hoc/CS2207.26.1.CH.02/.venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (120) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/nguyenkz/Documents/code/hoc/CS2207.26.1.CH.02/.venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochas

In [ ]:
config = {
    'version': 1,
    'random_seed': RANDOM_SEED,
    'dataset_file': DATASET_PATH.name,
    'input_shape': [8, 8],
    'feature_count': 64,
    'class_count': 10,
    'fit_indices': train_indices.astype(int).tolist(),
    'validation_indices': validation_indices.astype(int).tolist(),
    'test_indices': np.arange(len(y_test)).astype(int).tolist(),
    'scaler': {'mean': scaler.mean_.tolist(), 'std': scaler.scale_.tolist()},
    'models': MODEL_CONFIGS,
    'search': {
        'sample_count': int(search_count),
        'logistic_rounds': len(logistic_scores),
        'mlp_rounds': len(mlp_scores),
        'logistic_candidates': logistic_scores,
        'mlp_candidates': mlp_scores,
        'selected_logistic_validation_accuracy': best_logistic['validation_accuracy'],
        'selected_mlp_validation_accuracy': best_mlp['validation_accuracy'],
    },
}
CONFIG_PATH.write_text(json.dumps(config, indent=2), encoding='utf-8')
print(f'Wrote {CONFIG_PATH}')
assert len(logistic_scores) == 30
assert len(mlp_scores) == 30